# Resampling of Loop Study Public Dataset

In [4]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

### Read Existing Tables

In [63]:
file_path = "../../data/out/Loop study public dataset 2023-01-31/"
file_map = {
    'cgm': 'Loop_cgm_history.csv.gz',
    'bolus': 'Loop_bolus_event_history.csv.gz',
    'basal': 'Loop_basal_event_history.csv.gz',
}

In [99]:
def get_df_from_file(file_path, file_name, parse_datetime=True, sep=',', encoding='utf-8'):
    df = pd.read_csv(file_path + file_name, sep=sep, on_bad_lines='skip', encoding=encoding)
    if parse_datetime:
        df['date'] = pd.to_datetime(df['datetime'], unit='s')
    return df

In [65]:
def get_extended_df(original_df, value_column):
    """
    Get a df where quantities are distributed throughout 5-minute intervals instead of having start- and end dates.
    """
    new_rows = []
    for _, row in original_df.iterrows():
        new_rows.extend(split_duration(row, value_column))
    extended_df = pd.DataFrame(new_rows)
    extended_df.set_index('date', inplace=True)
    return extended_df

def split_duration(row, value_column):
    """
    For features with a duration, we split the values across 5-minute intervals by adding
    new rows for every 5-minute window in duration, and equally split the original quantity across those rows.
    """
    duration = row['end_date'] - row['date']
    rounded_duration = round(duration / pd.Timedelta(minutes=5)) * pd.Timedelta(minutes=5)
    num_intervals = rounded_duration // pd.Timedelta(minutes=5)
    if num_intervals < 1:
        num_intervals = 1
    value_per_interval = row[value_column] / num_intervals
    new_rows = []
    for i in range(int(num_intervals)):
        new_row = {
            'date': row['date'] + pd.Timedelta(minutes=5 * i),
            value_column: value_per_interval,
            'patient_id': row['patient_id'],
        }
        new_rows.append(new_row)
    return new_rows

In [66]:
df_glucose = get_df_from_file(file_path, file_map['cgm'])
df_bolus = get_df_from_file(file_path, file_map['bolus'])
df_basal = get_df_from_file(file_path, file_map['basal'])


### Resample Existing Tables

In [67]:
df_glucose.set_index('date', inplace=True)
df_glucose.head()

,datetime,cgm,patient_id
date,,,
2018-06-15 19:01:43,1529089303,396,3
2018-06-15 19:06:44,1529089604,394,3
2018-06-15 19:11:44,1529089904,387,3
2018-06-15 19:21:44,1529090504,338,3
2018-06-15 19:26:48,1529090808,331,3


In [68]:
df_bolus_orig = df_bolus.copy()
df_bolus['end_date'] = df_bolus['date'] + pd.to_timedelta(df_bolus['delivery_duration'], unit='s')
df_bolus = get_extended_df(df_bolus, 'bolus')
df_bolus

,bolus,patient_id
date,,
2017-10-20 10:38:25,2.70,960
2017-10-20 17:00:39,3.40,960
2017-10-20 20:09:03,3.50,960
2017-10-21 08:36:23,0.90,960
2017-10-21 09:42:10,1.70,960
...,...,...
2020-03-29 16:28:13,1.65,642
2020-03-29 17:24:23,1.30,642
2020-03-29 19:00:46,0.50,642


In [69]:
print(f'New sum after distribution of extended boluses: {df_bolus["bolus"].sum():.2f}, should be: {df_bolus_orig["bolus"].sum():.2f}')

New sum after distribution of extended boluses: 4668918.42, should be: 4668918.42


In [70]:
df_basal_orig = df_basal.copy()
df_basal.sort_values(by=['patient_id', 'date'], inplace=True)
df_basal.set_index('date', inplace=True)
df_basal

,patient_id,datetime,basal_rate
date,,,
2018-09-08 16:16:47,3,1536423407,0.410
2018-09-08 17:30:00,3,1536427800,0.467
2018-09-08 19:00:00,3,1536433200,0.550
2018-09-08 21:00:00,3,1536440400,0.500
2018-09-08 23:00:00,3,1536447600,0.333
...,...,...,...
2018-11-28 18:01:22,1212,1543428082,3.399
2018-11-28 18:02:15,1212,1543428135,3.790
2018-11-28 18:06:13,1212,1543428373,4.040


In [71]:
processed_dfs = []
subject_ids = df_glucose['patient_id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_glucose[df_glucose['patient_id'] == subject_id].copy()
    df_subject = df_subject[['cgm']].resample('5min', label='right').mean()
    df_subject['patient_id'] = subject_id
    df_subject.sort_index(inplace=True)

    def merge_data(df_col, df_subject, col_names, subject_id, agg_type='sum'):
        """ agg_type is data aggregation type. """
        df_subset = df_col[df_col['patient_id'] == subject_id].copy()
        if not df_subset.empty:
            if agg_type == 'mean':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').mean()
            elif agg_type == 'sum':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            elif agg_type == 'first':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').first()
            elif agg_type == 'last':
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').last()
            else:
                df_subset = df_subset[df_subset[col_names].notna()][col_names].resample('5min', label='right').sum()
            df_subject = pd.merge(df_subject, df_subset, on="date", how='outer')
        else:
            df_subject[col_names] = np.nan
        return df_subject

    # Add insulin and insulin type
    df_subject = merge_data(df_bolus, df_subject, ['bolus'], subject_id, agg_type='sum')
    df_subject = merge_data(df_basal, df_subject, ['basal_rate'], subject_id, agg_type='last')
    df_subject['basal_rate'] = df_subject['basal_rate'].ffill()
    
    df_subject['patient_id'] = subject_id
    df_subject = df_subject.rename(columns={'patient_id': 'id', 'basal_rate': 'basal', 'cgm': 'CGM'})
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)

Subjects: 851
3 is finished processing
4 is finished processing
5 is finished processing
6 is finished processing
7 is finished processing
9 is finished processing
10 is finished processing
11 is finished processing
12 is finished processing
13 is finished processing
15 is finished processing
16 is finished processing
18 is finished processing
19 is finished processing
20 is finished processing
21 is finished processing
24 is finished processing
25 is finished processing
26 is finished processing
27 is finished processing
30 is finished processing
32 is finished processing
33 is finished processing
34 is finished processing
38 is finished processing
41 is finished processing
43 is finished processing
44 is finished processing
45 is finished processing
46 is finished processing
47 is finished processing
50 is finished processing
52 is finished processing
53 is finished processing
55 is finished processing
57 is finished processing
59 is finished processing
60 is finished processing
62 i

In [72]:
df_final

,CGM,id,bolus,basal
date,,,,
2018-06-15 19:05:00,396.0,3,NaN,NaN
2018-06-15 19:10:00,394.0,3,NaN,NaN
2018-06-15 19:15:00,387.0,3,NaN,NaN
2018-06-15 19:20:00,NaN,3,NaN,NaN
2018-06-15 19:25:00,338.0,3,NaN,NaN
...,...,...,...,...
2018-11-28 18:40:00,141.0,1212,NaN,0.0
2018-11-28 18:45:00,128.0,1212,NaN,0.0
2018-11-28 18:50:00,125.0,1212,NaN,0.0


### Add Additional Tables

We found:
- Carbs
- Exercise
- Insulin type
- Age
- Weight
- Height
- Gender

In [73]:
raw_data_file_path = "../../data/raw/Loop study public dataset 2023-01-31/Data Tables/"

In [74]:
# Get time zone shift for subjects
df_time_zone = get_df_from_file(raw_data_file_path, 'PtRoster.txt', parse_datetime=False, sep='|')
df_time_zone = df_time_zone[['PtID', 'PtTimezoneOffset']]
df_time_zone = df_time_zone.rename(columns={'PtID': 'id', 'PtTimezoneOffset': 'tz_offset'})
df_time_zone

,id,tz_offset
0,963,-8.0
1,55,-8.0
2,1082,-5.0
3,1173,-8.0
4,296,-8.0
...,...,...
914,913,-5.0
915,459,-6.0
916,409,-5.0
917,854,-5.0


In [75]:
# No duplicate ids in this df
df_time_zone[df_time_zone.duplicated('id', keep=False)].sort_values(by=['id'])

,id,tz_offset


In [76]:
def get_df_with_local_date(df, utc_date_col, df_time_zone):
    processed_dfs = []
    subject_ids = df['id'].unique()
    print("Subjects:", len(subject_ids))
    for subject_id in subject_ids:
        df_subject = df[df['id'] == subject_id].copy()
        df_tz_subject = df_time_zone[df_time_zone['id'] == subject_id].copy()
        if not df_tz_subject.empty:
            subject_dt_offset = df_tz_subject['tz_offset'].iloc[0]
        else:
            subject_dt_offset = 0
        df_subject['date'] = df_subject[utc_date_col] + pd.to_timedelta(subject_dt_offset, unit='hour')
        processed_dfs.append(df_subject)
        print(f"{subject_id} is finished processing")
    return pd.concat(processed_dfs)

In [77]:
# Adding carbs
df_carbs = get_df_from_file(raw_data_file_path, 'LOOPDeviceFood.txt', parse_datetime=False, sep='|')
df_carbs = df_carbs.rename(columns={'PtID': 'id'})
df_carbs['UTCDtTm'] = pd.to_datetime(df_carbs['UTCDtTm'])
df_carbs = get_df_with_local_date(df_carbs, 'UTCDtTm', df_time_zone)
df_carbs[['id', 'UTCDtTm', 'date', 'CarbsNet']].head()

/var/folders/q1/5jqy1fgs07j9ptpmdsmmhxnw0000gn/T/ipykernel_93979/2272110270.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path + file_name, sep=sep)


Subjects: 837
1082 is finished processing
1173 is finished processing
296 is finished processing
1152 is finished processing
1013 is finished processing
829 is finished processing
104 is finished processing
942 is finished processing
105 is finished processing
960 is finished processing
467 is finished processing
458 is finished processing
589 is finished processing
407 is finished processing
856 is finished processing
605 is finished processing
738 is finished processing
787 is finished processing
691 is finished processing
763 is finished processing
653 is finished processing
511 is finished processing
85 is finished processing
164 is finished processing
425 is finished processing
213 is finished processing
1080 is finished processing
153 is finished processing
621 is finished processing
462 is finished processing
543 is finished processing
607 is finished processing
826 is finished processing
67 is finished processing
961 is finished processing
970 is finished processing
456 is fini

,id,UTCDtTm,date,CarbsNet
0,1082,2018-06-03 14:33:05,2018-06-03 09:33:05,5.0
1,1082,2018-06-03 14:00:23,2018-06-03 09:00:23,30.0
2,1082,2018-06-03 03:50:28,2018-06-02 22:50:28,25.0
3,1082,2018-06-03 00:10:57,2018-06-02 19:10:57,48.0
4,1082,2018-06-02 22:40:11,2018-06-02 17:40:11,30.0


In [78]:
df_carbs[['id', 'UTCDtTm', 'date', 'CarbsNet']]

,id,UTCDtTm,date,CarbsNet
0,1082,2018-06-03 14:33:05,2018-06-03 09:33:05,5.0
1,1082,2018-06-03 14:00:23,2018-06-03 09:00:23,30.0
2,1082,2018-06-03 03:50:28,2018-06-02 22:50:28,25.0
3,1082,2018-06-03 00:10:57,2018-06-02 19:10:57,48.0
4,1082,2018-06-02 22:40:11,2018-06-02 17:40:11,30.0
...,...,...,...,...
1004782,45,2019-03-18 12:32:27,2019-03-18 07:32:27,15.0
1004783,45,2019-03-18 01:56:25,2019-03-17 20:56:25,15.0
1004784,45,2019-03-17 19:07:40,2019-03-17 14:07:40,35.0
1352827,516,2019-04-24 02:33:18,2019-04-23 18:33:18,2.0


In [79]:
df_time_zone[df_time_zone['id'] == 45]

,id,tz_offset
333,45,-5.0


In [80]:
# Local time zone shifts look like they are working as expected

In [81]:
# Unit is always in grams
df_carbs[df_carbs['CarbsNet'] > 0]['CarbUnits'].unique()

array(['grams'], dtype=object)

In [82]:
df_carbs = df_carbs[['id', 'date', 'CarbsNet']]
df_carbs = df_carbs.rename(columns={'CarbsNet': 'carbs'})
df_carbs = df_carbs[df_carbs['carbs'] > 0]
df_carbs

,id,date,carbs
0,1082,2018-06-03 09:33:05,5.0
1,1082,2018-06-03 09:00:23,30.0
2,1082,2018-06-02 22:50:28,25.0
3,1082,2018-06-02 19:10:57,48.0
4,1082,2018-06-02 17:40:11,30.0
...,...,...,...
1004782,45,2019-03-18 07:32:27,15.0
1004783,45,2019-03-17 20:56:25,15.0
1004784,45,2019-03-17 14:07:40,35.0
1352827,516,2019-04-23 18:33:18,2.0


In [83]:
# Add carbs to df final
processed_dfs = []
subject_ids = df_final['id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_final[df_final['id'] == subject_id].copy()
    df_subject.sort_index(inplace=True)

    # Add carbs
    df_subject_carbs = df_carbs[df_carbs['id'] == subject_id].copy()
    df_subject_carbs.set_index('date', inplace=True)
    df_subject_carbs = df_subject_carbs[df_subject_carbs['carbs'].notna()]['carbs'].resample('5min', label='right').sum()
    df_subject = pd.merge(df_subject, df_subject_carbs, on="date", how='outer')
    
    df_subject['id'] = subject_id
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)
df_final

Subjects: 851
3 is finished processing
4 is finished processing
5 is finished processing
6 is finished processing
7 is finished processing
9 is finished processing
10 is finished processing
11 is finished processing
12 is finished processing
13 is finished processing
15 is finished processing
16 is finished processing
18 is finished processing
19 is finished processing
20 is finished processing
21 is finished processing
24 is finished processing
25 is finished processing
26 is finished processing
27 is finished processing
30 is finished processing
32 is finished processing
33 is finished processing
34 is finished processing
38 is finished processing
41 is finished processing
43 is finished processing
44 is finished processing
45 is finished processing
46 is finished processing
47 is finished processing
50 is finished processing
52 is finished processing
53 is finished processing
55 is finished processing
57 is finished processing
59 is finished processing
60 is finished processing
62 i

,CGM,id,bolus,basal,carbs
date,,,,,
2018-06-15 19:05:00,396.0,3,NaN,NaN,NaN
2018-06-15 19:10:00,394.0,3,NaN,NaN,NaN
2018-06-15 19:15:00,387.0,3,NaN,NaN,NaN
2018-06-15 19:20:00,NaN,3,NaN,NaN,NaN
2018-06-15 19:25:00,338.0,3,NaN,NaN,NaN
...,...,...,...,...,...
2018-11-28 18:40:00,141.0,1212,NaN,0.0,NaN
2018-11-28 18:45:00,128.0,1212,NaN,0.0,NaN
2018-11-28 18:50:00,125.0,1212,NaN,0.0,NaN


In [84]:
print(f'New sum after distribution of carbs: {df_final["carbs"].sum():.2f}, should be: {df_carbs["carbs"].sum():.2f}')

New sum after distribution of carbs: 39130874.69, should be: 39130874.69


In [85]:
# Adding exercise label and duration
df_exercise = get_df_from_file(raw_data_file_path, 'LOOPDeviceExercise.txt', parse_datetime=False, sep='|')
df_exercise

/var/folders/q1/5jqy1fgs07j9ptpmdsmmhxnw0000gn/T/ipykernel_93979/2272110270.py:2: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path + file_name, sep=sep)


,PtID,RecID,ParentLOOPDeviceUploadsID,DeviceDtTm,UTCDtTm,ExerciseName,DistanceValue,DistanceUnits,DurationValue,DurationUnits,...,TmZnOffset,OriginName,OriginVers,OriginType,OriginDeviceFirmwrVer,OriginDeviceHardwrVer,OriginDeviceManufact,OriginDeviceModel,OriginOperatingSystVer,OriginProductType
0,1173,1,23177,NaN,2018-09-16 18:34:46,Hiking - 4.37 miles,4.370740,miles,7213.940,seconds,...,NaN,com.apple.HealthKit,5.1.1,service,NaN,"Watch4,2",Apple,Watch,5.1.1,"Watch4,2"
1,1173,2,23177,NaN,2018-09-10 01:31:03,Walking - 0.87 miles,0.866911,miles,639.775,seconds,...,NaN,com.apple.HealthKit,0,service,NaN,NaN,NaN,NaN,5.1.1,"Watch4,2"
2,1173,3,23177,NaN,2018-09-14 18:55:57,Hiking - 4.73 miles,4.728440,miles,8383.710,seconds,...,NaN,com.apple.HealthKit,5.1.1,service,NaN,"Watch4,2",Apple,Watch,5.1.1,"Watch4,2"
3,1173,4,23177,NaN,2018-09-13 22:36:54,Walking - 1.63 miles,1.633190,miles,3206.160,seconds,...,NaN,com.apple.HealthKit,0,service,NaN,NaN,NaN,NaN,5.1.1,"Watch4,2"
4,1173,5,23177,NaN,2018-09-11 18:14:12,Walking - 0.78 miles,0.776254,miles,538.781,seconds,...,NaN,com.apple.HealthKit,0,service,NaN,NaN,NaN,NaN,5.1.1,"Watch4,2"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51723,760,55828,1414404,NaN,2019-03-14 21:49:59,Walking - 0.19 miles,0.189719,miles,1806.850,seconds,...,NaN,com.apple.HealthKit,4.3.2,service,NaN,NaN,NaN,NaN,4.3.2,"Watch1,1"
51724,760,55829,1414404,NaN,2019-01-28 17:05:24,Walking - 1.19 miles,1.191470,miles,1856.310,seconds,...,NaN,com.apple.HealthKit,4.3.2,service,NaN,NaN,NaN,NaN,4.3.2,"Watch1,1"
51725,853,55830,1469063,NaN,2019-11-10 18:23:51,Walking - 0.95 miles,0.951186,miles,1291.970,seconds,...,NaN,com.apple.HealthKit,5.3.2,service,NaN,"Watch2,3",Apple Inc.,Watch,5.3.2,"Watch2,3"
51726,853,55831,1469063,NaN,2020-02-15 18:02:45,Walking - 1.43 miles,1.432730,miles,3567.880,seconds,...,NaN,com.apple.HealthKit,6.1.3,service,NaN,"Watch2,3",Apple Inc.,Watch,6.1.3,"Watch2,3"


In [86]:
print(df_exercise['ExerciseName'].unique())
print(df_exercise['EnergyUnits'].unique())

['Hiking - 4.37 miles' 'Walking - 0.87 miles' 'Hiking - 4.73 miles' ...
 'Dance - 0.41 miles' 'Walking - 13.36 miles' 'Cycling - 2.32 miles']
['kilocalories' nan]


In [87]:
for val in df_exercise['ExerciseName'].unique():
    print(val)

Hiking - 4.37 miles
Walking - 0.87 miles
Hiking - 4.73 miles
Walking - 1.63 miles
Walking - 0.78 miles
Walking - 0.73 miles
Walking - 0.88 miles
Walking - 0.85 miles
Walking - 0.00 miles
Walking - 0.70 miles
Walking - 1.03 miles
Walking - 1.67 miles
Walking - 1.34 miles
Walking - 0.93 miles
Other Activity
Yoga
Walking - 1.62 miles
Walking - 0.74 miles
Running - 4.38 miles
Running - 4.21 miles
Walking - 1.13 miles
Running - 3.24 miles
Walking - 1.52 miles
Running - 4.29 miles
Running - 4.31 miles
High Intensity Interval Training
Running - 0.56 miles
Running - 0.72 miles
Rowing - 0.62 miles
Running - 1.19 miles
Running - 1.18 miles
Running - 3.00 miles
Cycling - 0.00 miles
Running - 2.43 miles
Cycling - 12.00 miles
Swimming - 0.62 miles
Other Activity - 0.00 miles
Running - 3.08 miles
Running - 4.63 miles
Running - 3.95 miles
Running - 5.89 miles
Running - 4.00 miles
Running - 3.61 miles
Running - 4.44 miles
Running - 4.62 miles
Walking - 0.79 miles
Walking - 0.46 miles
Walking - 3.88 mi

In [88]:
# Adding exercise label and duration
df_exercise['workout_label'] = df_exercise['ExerciseName'].str.split(' -').str[0]
df_exercise['workout_duration'] = df_exercise['DurationValue'] / 60
df_exercise['calories_burned'] = df_exercise['EnergyValue']
df_exercise = df_exercise.rename(columns={'PtID': 'id'})
df_exercise['UTCDtTm'] = pd.to_datetime(df_exercise['UTCDtTm'])
df_exercise = get_df_with_local_date(df_exercise, 'UTCDtTm', df_time_zone)
df_exercise = df_exercise[['id', 'date', 'workout_label', 'workout_duration', 'calories_burned']]
df_exercise

Subjects: 493
1173 is finished processing
296 is finished processing
1013 is finished processing
829 is finished processing
104 is finished processing
105 is finished processing
467 is finished processing
589 is finished processing
407 is finished processing
605 is finished processing
738 is finished processing
787 is finished processing
511 is finished processing
85 is finished processing
621 is finished processing
462 is finished processing
67 is finished processing
456 is finished processing
970 is finished processing
1081 is finished processing
968 is finished processing
644 is finished processing
859 is finished processing
9 is finished processing
242 is finished processing
388 is finished processing
84 is finished processing
432 is finished processing
508 is finished processing
1093 is finished processing
533 is finished processing
438 is finished processing
998 is finished processing
295 is finished processing
932 is finished processing
1211 is finished processing
1105 is finish

,id,date,workout_label,workout_duration,calories_burned
0,1173,2018-09-16 10:34:46,Hiking,120.232333,461.3690
1,1173,2018-09-09 17:31:03,Walking,10.662917,80.9740
2,1173,2018-09-14 10:55:57,Hiking,139.728500,624.4150
3,1173,2018-09-13 14:36:54,Walking,53.436000,166.4610
4,1173,2018-09-11 10:14:12,Walking,8.979683,92.7650
...,...,...,...,...,...
51276,450,2019-09-08 18:58:38,Elliptical,28.102667,155.3220
51277,450,2019-09-05 08:31:08,Walking,37.529500,114.4010
51278,450,2019-08-27 14:07:59,Walking,23.511500,82.3137
51576,200,2019-07-26 14:34:36,Functional Strength Training,38.200000,181.3640


In [89]:
df_exercise[['workout_duration', 'calories_burned']] = df_exercise[['workout_duration', 'calories_burned']].round()
df_exercise

,id,date,workout_label,workout_duration,calories_burned
0,1173,2018-09-16 10:34:46,Hiking,120.0,461.0
1,1173,2018-09-09 17:31:03,Walking,11.0,81.0
2,1173,2018-09-14 10:55:57,Hiking,140.0,624.0
3,1173,2018-09-13 14:36:54,Walking,53.0,166.0
4,1173,2018-09-11 10:14:12,Walking,9.0,93.0
...,...,...,...,...,...
51276,450,2019-09-08 18:58:38,Elliptical,28.0,155.0
51277,450,2019-09-05 08:31:08,Walking,38.0,114.0
51278,450,2019-08-27 14:07:59,Walking,24.0,82.0
51576,200,2019-07-26 14:34:36,Functional Strength Training,38.0,181.0


In [90]:
# Add workout label and duration to the df_final
processed_dfs = []
subject_ids = df_final['id'].unique()
print("Subjects:", len(subject_ids))
for subject_id in subject_ids:
    df_subject = df_final[df_final['id'] == subject_id].copy()
    df_subject.sort_index(inplace=True)

    df_subject_workouts = df_exercise[df_exercise['id'] == subject_id].copy()
    df_subject_workouts.set_index('date', inplace=True)
    df_subject_calburn = df_subject_workouts[['calories_burned']].resample('5min', label='right').sum()
    df_subject_workouts = df_subject_workouts[['workout_label', 'workout_duration']].resample('5min', label='right').last()
    df_subject = pd.merge(df_subject, df_subject_calburn, on="date", how='outer')
    df_subject = pd.merge(df_subject, df_subject_workouts, on="date", how='outer')

    df_subject['id'] = subject_id
    processed_dfs.append(df_subject)
    print(f"{subject_id} is finished processing")

df_final = pd.concat(processed_dfs)
df_final

Subjects: 851
3 is finished processing
4 is finished processing
5 is finished processing
6 is finished processing
7 is finished processing
9 is finished processing
10 is finished processing
11 is finished processing
12 is finished processing
13 is finished processing
15 is finished processing
16 is finished processing
18 is finished processing
19 is finished processing
20 is finished processing
21 is finished processing
24 is finished processing
25 is finished processing
26 is finished processing
27 is finished processing
30 is finished processing
32 is finished processing
33 is finished processing
34 is finished processing
38 is finished processing
41 is finished processing
43 is finished processing
44 is finished processing
45 is finished processing
46 is finished processing
47 is finished processing
50 is finished processing
52 is finished processing
53 is finished processing
55 is finished processing
57 is finished processing
59 is finished processing
60 is finished processing
62 i

,CGM,id,bolus,basal,carbs,calories_burned,workout_label,workout_duration
date,,,,,,,,
2018-06-15 19:05:00,396.0,3,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-15 19:10:00,394.0,3,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-15 19:15:00,387.0,3,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-15 19:20:00,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN
2018-06-15 19:25:00,338.0,3,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2018-11-28 18:40:00,141.0,1212,NaN,0.0,NaN,NaN,NaN,NaN
2018-11-28 18:45:00,128.0,1212,NaN,0.0,NaN,NaN,NaN,NaN
2018-11-28 18:50:00,125.0,1212,NaN,0.0,NaN,NaN,NaN,NaN


In [91]:
df_final[df_final['workout_label'].notna()]

,CGM,id,bolus,basal,carbs,calories_burned,workout_label,workout_duration
date,,,,,,,,
2018-12-02 15:55:00,89.0,3,0.00,0.374,0.0,42.0,Running,25.0
2018-12-15 14:35:00,121.0,3,0.00,0.000,0.0,7.0,Cycling,11.0
2018-12-15 17:20:00,90.0,3,0.00,0.000,0.0,4.0,Cycling,3.0
2018-12-15 17:25:00,96.0,3,0.00,0.000,0.0,1.0,Cycling,7.0
2018-12-15 18:05:00,160.0,3,0.00,0.399,0.0,13.0,Cycling,10.0
...,...,...,...,...,...,...,...,...
2019-12-19 08:20:00,140.0,1211,0.95,1.966,0.0,288.0,Rowing,50.0
2019-12-25 10:00:00,227.0,1211,0.00,0.203,0.0,565.0,Hiking,126.0
2019-12-25 13:35:00,177.0,1211,0.00,1.987,0.0,45.0,Rowing,10.0


In [92]:
print(f'New sum after distribution of calburn: {df_final["calories_burned"].sum():.2f}, should be: {df_exercise["calories_burned"].sum():.2f}')

New sum after distribution of calburn: 14932794.00, should be: 14932794.00


In [101]:
df_insulin_type = get_df_from_file(raw_data_file_path, 'LOOPDeviceIssueRpt.txt', parse_datetime=False, sep='|', encoding='latin1')
df_insulin_type = df_insulin_type[['PtID', 'InsulinModel']]
df_insulin_type = df_insulin_type.rename(columns={'PtID': 'id', 'InsulinModel': 'insulin_type'})
df_insulin_type

,id,insulin_type
0,963,(humalogNovologAdult(ExponentialInsulinModel(a...
1,963,(humalogNovologAdult(ExponentialInsulinModel(a...
2,963,(humalogNovologAdult(ExponentialInsulinModel(a...
3,963,(humalogNovologAdult(ExponentialInsulinModel(a...
4,963,(WalshInsulinModel(actionDuration: 18000.0))
...,...,...
7131,339,(humalogNovologChild(ExponentialInsulinModel(a...
7132,124,(humalogNovologAdult(ExponentialInsulinModel(a...
7133,1121,(humalogNovologChild(ExponentialInsulinModel(a...
7134,436,(humalogNovologAdult(ExponentialInsulinModel(a...


In [102]:
df_insulin_type['insulin_type'].unique()

array(['(humalogNovologAdult(ExponentialInsulinModel(actionDuration: 21600.0, peakActivityTime: 4500.0, initialDelay: 1200.0)',
       '(humalogNovologAdult(ExponentialInsulinModel(actionDuration: 21600.0, peakActivityTime: 4500.0))',
       '(humalogNovologAdult(ExponentialInsulinModel(actionDuration: 21600.0, peakActivityTime: 4500.0, initialDelay: 1200.0))',
       '(WalshInsulinModel(actionDuration: 18000.0))',
       '(WalshInsulinModel(actionDuration: 21600.0))',
       '(fiasp(ExponentialInsulinModel(actionDuration: 21600.0, peakActivityTime: 3300.0, initialDelay: 600.0)',
       '(fiasp(ExponentialInsulinModel(actionDuration: 21600.0, peakActivityTime: 3300.0))',
       '(humalogNovologChild(ExponentialInsulinModel(actionDuration: 21600.0, peakActivityTime: 3900.0, initialDelay: 1200.0)',
       '(humalogNovologChild(ExponentialInsulinModel(actionDuration: 21600.0, peakActivityTime: 3900.0, initialDelay: 1200.0))',
       '(humalogNovologAdult(ExponentialInsulinModel(actionDura

In [107]:
df_insulin_type['insulin_type'] = df_insulin_type['insulin_type'].str.split('(').str[1]
df_insulin_type = df_insulin_type[df_insulin_type['insulin_type'].notna()]
df_insulin_type

/var/folders/q1/5jqy1fgs07j9ptpmdsmmhxnw0000gn/T/ipykernel_93979/79857320.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_insulin_type['insulin_type'] = df_insulin_type['insulin_type'].str.split('(').str[1]


,id,insulin_type,insulin_type_formatted
0,963,humalogNovologAdult,humalogNovologAdult
1,963,humalogNovologAdult,humalogNovologAdult
2,963,humalogNovologAdult,humalogNovologAdult
3,963,humalogNovologAdult,humalogNovologAdult
4,963,WalshInsulinModel,WalshInsulinModel
...,...,...,...
7131,339,humalogNovologChild,humalogNovologChild
7132,124,humalogNovologAdult,humalogNovologAdult
7133,1121,humalogNovologChild,humalogNovologChild
7134,436,humalogNovologAdult,humalogNovologAdult


In [108]:
def add_single_value_to_subjects(df, df_new_val, col_name):
    # TODO: this function is very inefficient... add value directly to located rows instead
    processed_dfs = []
    subject_ids = df['id'].unique()
    for subject_id in subject_ids:
        df_subject = df[df['id'] == subject_id].copy()
        df_subject.sort_index(inplace=True)
    
        user_data = df_new_val[df_new_val['id'] == subject_id].copy()
        if not user_data.empty:
            df_subject[col_name] = user_data[col_name].iloc[0]
        else:
            df_subject[col_name] = np.nan        
        processed_dfs.append(df_subject)
        
    df = pd.concat(processed_dfs)
    return df

In [109]:
# Add insulin type
df_final = add_single_value_to_subjects(df_final, df_insulin_type, 'insulin_type')
df_final

,CGM,id,bolus,basal,carbs,calories_burned,workout_label,workout_duration,insulin_type
date,,,,,,,,,
2018-06-15 19:05:00,396.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild
2018-06-15 19:10:00,394.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild
2018-06-15 19:15:00,387.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild
2018-06-15 19:20:00,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild
2018-06-15 19:25:00,338.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild
...,...,...,...,...,...,...,...,...,...
2018-11-28 18:40:00,141.0,1212,NaN,0.0,NaN,NaN,NaN,NaN,humalogNovologAdult
2018-11-28 18:45:00,128.0,1212,NaN,0.0,NaN,NaN,NaN,NaN,humalogNovologAdult
2018-11-28 18:50:00,125.0,1212,NaN,0.0,NaN,NaN,NaN,NaN,humalogNovologAdult


In [111]:
# Add age
df_age = get_df_from_file(raw_data_file_path, 'PtRoster.txt', parse_datetime=False, sep='|')
df_age = df_age[['PtID', 'AgeAtEnrollment']]
df_age = df_age.rename(columns={'PtID': 'id', 'AgeAtEnrollment': 'age'})
df_age

,id,age
0,963,16
1,55,34
2,1082,29
3,1173,23
4,296,47
...,...,...
914,913,10
915,459,6
916,409,9
917,854,11


In [112]:
# Add age
df_final = add_single_value_to_subjects(df_final, df_age, 'age')
df_final.head()

,CGM,id,bolus,basal,carbs,calories_burned,workout_label,workout_duration,insulin_type,age
date,,,,,,,,,,
2018-06-15 19:05:00,396.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8
2018-06-15 19:10:00,394.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8
2018-06-15 19:15:00,387.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8
2018-06-15 19:20:00,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8
2018-06-15 19:25:00,338.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8


In [123]:
# Add gender
df_user_data = get_df_from_file(raw_data_file_path, 'Surveys.txt', parse_datetime=False, sep='|')
df_user_data = df_user_data[['SubjectID', 'gender', 'weight', 'height_feet', 'height_inches']]
df_user_data = df_user_data.dropna(how='all', subset=['gender', 'weight', 'height_feet', 'height_inches'])
# Weight in kg
df_user_data['weight'] = df_user_data.apply(
    lambda row: row['weight'] * 0.453592,
    axis=1
)
# Height in cm
df_user_data['height'] = df_user_data.apply(
    lambda row: (row['height_feet'] * 30.48) + (row['height_inches'] * 2.54),
    axis=1
)
# Gender from int to string
df_user_data['gender'] = df_user_data['gender'].map({1: 'Male', 2: 'Female', 3: 'Non-binary'})

df_user_data = df_user_data[['SubjectID', 'gender', 'weight', 'height']]
df_user_data[['weight', 'height']] = df_user_data[['weight', 'height']].round(1)
df_user_data = df_user_data.rename(columns={'SubjectID': 'id'})
df_user_data

,id,gender,weight,height
0,4,Male,99.8,198.1
2,5,Female,108.9,167.6
5,6,Male,59.0,175.3
8,7,Male,35.4,144.8
11,9,Female,74.8,157.5
...,...,...,...,...
2188,1208,Female,44.0,139.7
2191,1209,Male,14.7,91.4
2194,1210,Female,64.9,157.5
2197,1211,Female,70.3,167.6


In [124]:
# Add weight, height, and gender
df_final = add_single_value_to_subjects(df_final, df_user_data, 'weight')
df_final = add_single_value_to_subjects(df_final, df_user_data, 'height')
df_final = add_single_value_to_subjects(df_final, df_user_data, 'gender')
df_final.head()

,CGM,id,bolus,basal,carbs,calories_burned,workout_label,workout_duration,insulin_type,age,weight,height,gender
date,,,,,,,,,,,,,
2018-06-15 19:05:00,396.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:10:00,394.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:15:00,387.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:20:00,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:25:00,338.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN


### Save Resampled Data

In [125]:
df_final

,CGM,id,bolus,basal,carbs,calories_burned,workout_label,workout_duration,insulin_type,age,weight,height,gender
date,,,,,,,,,,,,,
2018-06-15 19:05:00,396.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:10:00,394.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:15:00,387.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:20:00,NaN,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
2018-06-15 19:25:00,338.0,3,NaN,NaN,NaN,NaN,NaN,NaN,humalogNovologChild,8,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-11-28 18:40:00,141.0,1212,NaN,0.0,NaN,NaN,NaN,NaN,humalogNovologAdult,12,65.8,152.4,Female
2018-11-28 18:45:00,128.0,1212,NaN,0.0,NaN,NaN,NaN,NaN,humalogNovologAdult,12,65.8,152.4,Female
2018-11-28 18:50:00,125.0,1212,NaN,0.0,NaN,NaN,NaN,NaN,humalogNovologAdult,12,65.8,152.4,Female


In [126]:
save_file_path = "../../data/resampled/"
os.makedirs(save_file_path, exist_ok=True)
df_final.to_csv(save_file_path + 'Loop.csv')

In [ ]:
# TODO: Split into smaller files???

In [127]:
# Split into smaller partitions, by ids so that a single id is always in a single file
save_file_path = "../../data/resampled/"
os.makedirs(save_file_path, exist_ok=True)
id_column = 'id'
unique_ids = df_final[id_column].unique()
np.random.shuffle(unique_ids) 
split = 8
id_splits = np.array_split(unique_ids, split)  # Split into 8 files

for i, id_subset in enumerate(id_splits, start=1):
    subset_df = df_final[df_final[id_column].isin(id_subset)]
    save_file_name = f"Loop_Part{i}_of_{split}.csv"
    subset_df.to_csv(f"{save_file_path}{save_file_name}", index=False)
    print(f"Saved {save_file_path}{save_file_name} with {len(subset_df)} rows")


Saved ../../data/resampled/Loop_Part1_of_8.csv with 11556695 rows
Saved ../../data/resampled/Loop_Part2_of_8.csv with 11970501 rows
Saved ../../data/resampled/Loop_Part3_of_8.csv with 12311910 rows
Saved ../../data/resampled/Loop_Part4_of_8.csv with 12115837 rows
Saved ../../data/resampled/Loop_Part5_of_8.csv with 12252085 rows
Saved ../../data/resampled/Loop_Part6_of_8.csv with 11948283 rows
Saved ../../data/resampled/Loop_Part7_of_8.csv with 12221154 rows
Saved ../../data/resampled/Loop_Part8_of_8.csv with 11977438 rows


In [128]:
total_rows_before = len(df_final)
total_rows_after = sum(len(df_final[df_final[id_column].isin(subset)]) for subset in id_splits)

print(f"Total rows before split: {total_rows_before}")
print(f"Total rows after split: {total_rows_after}")  


Total rows before split: 96353903
Total rows after split: 96353903


In [129]:
print(df_final[id_column].isna().sum())  # Check missing IDs


0


In [6]:
# Paths to files
full_file = "../../data/resampled/Loop.csv"
split_folder = "../../data/resampled/"
split_files = [f for f in os.listdir(split_folder) if f.startswith("Loop_Part") and f.endswith(".csv")]

# Column with unique subject IDs (change if needed)
id_column = "id"

# Read full file
df_full = pd.read_csv(full_file, low_memory=False)
total_rows_full = len(df_full)
unique_ids_full = df_full[id_column].nunique()

# Print comparison
print("📌 **Full File Stats**")
print(f"Total rows: {total_rows_full}")
print(f"Unique subjects: {unique_ids_full}")

# Read split files and merge row counts
total_rows_split = 0
unique_ids_split = set()

for file in split_files:
    df_part = pd.read_csv(os.path.join(split_folder, file), low_memory=False)
    total_rows_split += len(df_part)
    print("total_rows_split ", total_rows_split)
    unique_ids_split.update(df_part[id_column].unique())  # Store unique IDs

unique_ids_split_count = len(unique_ids_split)

print("\n📌 **Split Files Stats**")
print(f"Total rows across all parts: {total_rows_split}")
print(f"Unique subjects across all parts: {unique_ids_split_count}")

# Check if data matches
if total_rows_full == total_rows_split and unique_ids_full == unique_ids_split_count:
    print("\n✅ The split files correctly contain all data!")
else:
    print("\n⚠️ Data Mismatch! Some rows or subjects may be missing.")


📌 **Full File Stats**
Total rows: 96353903
Unique subjects: 851
total_rows_split  12221154
total_rows_split  24169437
total_rows_split  35726132
total_rows_split  47841969
total_rows_split  60094054
total_rows_split  72071492
total_rows_split  84383402
total_rows_split  96353903

📌 **Split Files Stats**
Total rows across all parts: 96353903
Unique subjects across all parts: 851

✅ The split files correctly contain all data!
